In [2]:
import os
import ast
import operator
from pathlib import Path

from google.colab import userdata

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

print("Libraries imported successfully!")

/tmp/ipykernel_2266/1236743122.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


Libraries imported successfully!


In [3]:
search_tool = DuckDuckGoSearchRun()

print("Web search tool created!")

Web search tool created!


In [4]:
search_result = search_tool.invoke("What is Generative AI?")

print(search_result)

Generative AI is artificial intelligence (AI) that can create original content in response to a user's prompt or request. Generative AI is a type of artificial intelligence designed to create new content such as text, images, music or even code by learning patterns from existing data. These models generate original outputs that are often indistinguishable from human-created content. These models use techniques like deep learning and neural networks to generate output. Generative AI is a popular technology that can quickly complete time-consuming tasks. Learn more about how it works, the benefits and limitations of AI generators, and jobs to explore if you're interested in this field. Generative AI is a kind of artificial intelligence technology that relies on deep learning models trained on large data sets to create new content. The meaning of GENERATIVE AI is artificial intelligence that is capable of generating new content (such as images or text) in response to a submitted prompt (s

In [5]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression such as 25 * 4 or (100 + 50) / 5."""

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos
    }

    def calculate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            left = calculate(node.left)
            right = calculate(node.right)

            if isinstance(node.op, ast.Pow) and abs(right) > 10:
                raise ValueError("Exponent is too large.")

            return allowed_operators[type(node.op)](left, right)

        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](calculate(node.operand))

        raise ValueError("Unsupported mathematical expression.")

    try:
        tree = ast.parse(expression, mode="eval")
        result = calculate(tree.body)
        return str(result)

    except Exception as error:
        return f"Calculation error: {error}"

print("Calculator tool created!")

Calculator tool created!


In [6]:
print(calculator.invoke({"expression": "250 * 4"}))
print(calculator.invoke({"expression": "(100 + 50) / 5"}))

1000
30.0


In [7]:
@tool
def file_reader(filename: str) -> str:
    """Read a text file from the current project folder."""

    allowed_files = {
        "project_notes.txt",
        "sample_document.txt"
    }

    if filename not in allowed_files:
        return "Error: This file is not in the allowed file list."

    path = Path(filename)

    if not path.is_file():
        return "Error: File not found."

    try:
        return path.read_text(encoding="utf-8")[:10000]

    except Exception as error:
        return f"File reading error: {error}"

print("File reader tool created!")

File reader tool created!


In [8]:
with open("project_notes.txt", "w") as file:
    file.write("""
    Project Name: Generative AI Internship

    Technologies:
    Python, LangChain, Gemini, ChromaDB and Streamlit.

    Project 1:
    Structured LLM Prompt Pipeline and JSON Extractor.

    Project 2:
    Document Question Answering Assistant using RAG.

    Project 3:
    Multi-Tool Autonomous AI Agent.
    """)

print("Sample file created successfully!")

Sample file created successfully!


In [9]:
print(file_reader.invoke({"filename": "project_notes.txt"}))


    Project Name: Generative AI Internship

    Technologies:
    Python, LangChain, Gemini, ChromaDB and Streamlit.

    Project 1:
    Structured LLM Prompt Pipeline and JSON Extractor.

    Project 2:
    Document Question Answering Assistant using RAG.

    Project 3:
    Multi-Tool Autonomous AI Agent.
    


In [10]:
tools = [
    search_tool,
    calculator,
    file_reader
]

print("Total tools available:", len(tools))

for item in tools:
    print("-", item.name)

Total tools available: 3
- duckduckgo_search
- calculator
- file_reader


In [11]:
from google.colab import userdata
import os

GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("Gemini model initialized successfully!")

Gemini model initialized successfully!


In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are an intelligent autonomous AI assistant.

    You have access to three tools:

    1. DuckDuckGo search: Use it to find information online.
    2. Calculator: Use it for mathematical calculations.
    3. File reader: Use it to read permitted local text files.

    Instructions:
    - Decide which tool is suitable for each request.
    - Use tools when they are needed.
    - Do not invent search results or file contents.
    - Explain your final answer clearly.
    - If a tool fails, explain the problem.
    """
)

print("Autonomous AI agent created successfully!")

Autonomous AI agent created successfully!


In [13]:
def ask_agent(question):

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    return response["messages"][-1].content

print("Agent chat function created!")

Agent chat function created!


In [14]:
answer = ask_agent(
    "Calculate 250 multiplied by 4 using the calculator tool."
)

print("AI AGENT RESPONSE:")
print(answer)

AI AGENT RESPONSE:
[{'type': 'text', 'text': '250 multiplied by 4 is **1,000**.', 'extras': {'signature': 'EqcBCqQBAWkUfRNJ1UEW4FmPBpeOHLzBrWR4X44uOAImQpicQgL1+xOFLKjupArNrzi3Ag4K5106b0dB030q7ll2eEmz6XfaAh4ufiMq71eBQiBqH4TKhFRop6LrzM+SXfbXsIgUDrvyj+xNU6UDVfacWstEHKsV7sXwyEgAnlvA57dwYXmH6KPbiXqmsv5nfRZZFB3oaO4ny6goJPKqADiytJzuqy54x+c='}}]


In [15]:
answer = ask_agent(
    "Search the web and explain what Generative AI is in simple words."
)

print(answer)

[{'type': 'text', 'text': '**Generative AI (Generative Artificial Intelligence)** is a type of AI that can create **new, original content**—such as text, images, music, videos, or computer code—based on simple instructions or prompts you give it.\n\n---\n\n### How Does It Work?\n\n1. **Learning from Examples:** Imagine reading millions of books or looking at millions of paintings. Over time, you learn how sentences flow, how colors blend, and what makes a story or a picture make sense. Generative AI does the same thing using massive amounts of data.\n2. **Recognizing Patterns:** It learns patterns, styles, grammar, and structures rather than simply memorizing answers.\n3. **Creating Something New:** When you ask it a question or give it a prompt (like *"Write a poem about space"* or *"Draw a cat wearing a astronaut helmet"*), it uses those learned patterns to generate a completely new piece of work that wasn\'t there before.\n\n---\n\n### Generative AI vs. Traditional AI\n\n* **Traditi

In [18]:
answer = ask_agent(
    "Read project_notes.txt and tell me which technologies are mentioned."
)

print(answer)

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 17h28m8.601745169s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': '

In [23]:
agent_logs = []

def run_agent(question):

    print("User:", question)

    try:
        response = agent.invoke({
            "messages": [
                {"role": "user", "content": question}
            ]
        })

        messages = response["messages"]

        # Record tool calls
        tool_calls = []

        for message in messages:
            if getattr(message, "tool_calls", None):
                for call in message.tool_calls:
                    tool_calls.append(call["name"])

        answer = messages[-1].content

        log = {
            "question": question,
            "tools_used": tool_calls,
            "answer": answer,
            "status": "success"
        }

        agent_logs.append(log)

        print("\nTools used:", tool_calls)
        print("\nAgent answer:", answer)

        return log

    except Exception as error:

        log = {
            "question": question,
            "tools_used": [],
            "answer": str(error),
            "status": "error"
        }

        agent_logs.append(log)

        print("\nAgent error:", error)

        return log

print("Agent logging system created!")

Agent logging system created!


In [24]:
print(calculator.invoke({"expression": "500 + 250"}))

750


In [25]:
print(file_reader.invoke({"filename": "project_notes.txt"}))


    Project Name: Generative AI Internship

    Technologies:
    Python, LangChain, Gemini, ChromaDB and Streamlit.

    Project 1:
    Structured LLM Prompt Pipeline and JSON Extractor.

    Project 2:
    Document Question Answering Assistant using RAG.

    Project 3:
    Multi-Tool Autonomous AI Agent.
    


In [26]:
print(search_tool.invoke("Latest developments in generative AI"))

Oct 19, 2026 · Explore the latest news and expert commentary on Generative AI, brought to you by the editors of AI Business Oct 12, 2026 · Explore Generative AI trends for 2026 and beyond - from AI-driven creativity to edge computing. Stay ahead with this Generative AI advancements! Jul 29, 2026 · This article serves as your one-stop destination for staying updated with the latest developments, breakthroughs, and trends in the ever-evolving field of Generative AI. Trends in AI are driven not only by advancements in AI models and algorithms themselves, but by the ever-expanding array of use cases to which generative AI (gen AI) capabilities are being applied. Jul 9, 2026 · Read the latest news and analysis from TechCrunch about the startups and founders behind Generative AI, and the impact on our world. Aug 11, 2026 · The latest AI advancements show a quick pace of progress, touching areas like healthcare and energy. A new chip from UC San Diego cuts down energy waste in data centers. A

In [27]:
run_agent("Calculate 500 multiplied by 25.")

User: Calculate 500 multiplied by 25.

Agent error: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 16h56m20.059407285s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', '

{'question': 'Calculate 500 multiplied by 25.',
 'tools_used': [],
 'answer': "Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\\nPlease retry in 16h56m20.059407285s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.co

In [28]:
run_agent("Search the web for the applications of generative AI.")

User: Search the web for the applications of generative AI.

Agent error: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 16h55m45.782734324s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/googl

{'question': 'Search the web for the applications of generative AI.',
 'tools_used': [],
 'answer': "Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\\nPlease retry in 16h55m45.782734324s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type

In [29]:
run_agent("Read project_notes.txt and list the technologies used.")

User: Read project_notes.txt and list the technologies used.

Agent error: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 16h54m54.73544101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/googl

{'question': 'Read project_notes.txt and list the technologies used.',
 'tools_used': [],
 'answer': "Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\\nPlease retry in 16h54m54.73544101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type

In [30]:
import json

with open("agent_execution_logs.json", "w") as file:
    json.dump(agent_logs, file, indent=4)

print("Agent execution logs saved!")
print("Total tests recorded:", len(agent_logs))

Agent execution logs saved!
Total tests recorded: 3
